In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os

# Set your path here
train_dir = "/content/drive/MyDrive/AI & ML/week6/FruitinAmazon"

# Loading data with a 20% validation split
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="both",
    seed=1337,
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical' # Important: matching the 'categorical_crossentropy' in the PDF
)

Found 120 files belonging to 2 classes.
Using 96 files for training.
Using 24 files for validation.


In [2]:
#task1
# Data Augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])

# Custom Model
model_scratch = keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(), # Added as per Task 1
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25), # Added as per Task 1

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(train_ds.class_names), activation='softmax')
])

model_scratch.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_scratch.summary() # Deliverable: Understand Model Summary

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 200704)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    25,690,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,710,274 (98.08 MB)

 Trainable params: 25,710,082 (98.08 MB)

 Non-trainable params: 192 (768.00 B)

In [3]:
#task2
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

# 1. Load Pre-trained Model
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Freeze the Layers
for layer in base_model.layers:
    layer.trainable = False

# 3. Add Custom Layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(len(train_ds.class_names), activation='softmax')(x)

# 4. Create and Compile Final Model
transfer_model = Model(inputs=base_model.input, outputs=predictions)
transfer_model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# 5. Fit the model
history = transfer_model.fit(train_ds, validation_data=val_ds, epochs=10)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 70s 24s/step - accuracy: 0.5833 - loss: 8.0962 - val_accuracy: 0.7083 - val_loss: 13.6933
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 53s 18s/step - accuracy: 0.7500 - loss: 5.8593 - val_accuracy: 0.3750 - val_loss: 3.0078
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 78s 18s/step - accuracy: 0.5625 - loss: 3.6516 - val_accuracy: 0.5000 - val_loss: 2.4745
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 84s 19s/step - accuracy: 0.8854 - loss: 0.9328 - val_accuracy: 0.7083 - val_loss: 7.7065
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 48s 17s/step - accuracy: 0.8021 - loss: 1.4640 - val_accuracy: 0.7083 - val_loss: 6.8314
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 49s 18s/step - accuracy: 0.9479 - loss: 0.2401 - val_accuracy: 0.6250 - val_loss: 4.0746
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 50s 18s/step - accuracy: 0.9271 - loss: 0.2664 - val_accuracy: 0.4583 - val_loss: 3.9536
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 81s 18s/step - accuracy: 0.8854 - los

In [6]:
import numpy as np
from sklearn.metrics import classification_report

# Get actual labels and predictions
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = transfer_model.predict(images)
    y_true.extend(np.argmax(labels, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

# Generate Classification Report
print(classification_report(y_true, y_pred, target_names=train_ds.class_names))

1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step
              precision    recall  f1-score   support

        test       0.50      0.14      0.22         7
       train       0.73      0.94      0.82        17

    accuracy                           0.71        24
   macro avg       0.61      0.54      0.52        24
weighted avg       0.66      0.71      0.65        24

